# 03 — Catastro Multipropósito IGAC para Urabá

## Objetivo
Cargar, explorar y vincular los datos del Catastro Multipropósito del IGAC para los municipios de la región de Urabá.
El catastro multipropósito provee geometrías de predios, clasificación de uso del suelo y datos de edificaciones
que se usan para desagregación dasimétrca de población.

## Fuentes de descarga
- **Portal IGAC Catastro Multipropósito**: https://www.igac.gov.co/es/contenido/areas-estrategicas/catastro-multiproposito
- **Geovisor IGAC**: https://geoportal.igac.gov.co/
- **Datos Abiertos IGAC**: https://datos.igac.gov.co/
- **Capa predios rurales/urbanos**: disponible como Shapefile o GeoJSON por municipio

## Municipios con cobertura disponible (Urabá)
| DIVIPOLA | Municipio | Estado catastro |
|----------|-----------|----------------|
| 05045 | Apartadó | Urbano disponible |
| 05147 | Chigorodó | Parcial |
| 05120 | Carepa | En proceso |
| 05837 | Turbo | Urbano disponible |
| 05544 | Necoclí | En proceso |
| 05665 | San Pedro de Urabá | Sin cobertura |
| 05659 | San Juan de Urabá | Sin cobertura |
| 05051 | Arboletes | Parcial |

> **Nota**: Descargar archivos en `data/raw/igac/` con la estructura:
> `data/raw/igac/predios/<DIVIPOLA>_predios.gpkg` y `data/raw/igac/edificaciones/<DIVIPOLA>_edif.gpkg`


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Asegurar que src/ esté en el path
ROOT = Path("../").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx
import warnings
warnings.filterwarnings("ignore")

# Módulos Atlas Urabá
from src.ingestion.dane_loader import cargar_mgn_manzanas
from src.processing.disaggregate import disaggregate_dasymetric

# Constantes
DIVIPOLA_APARTADO = "05045"
DATA_RAW_IGAC = ROOT / "data" / "raw" / "igac"
DATA_RAW_DANE = ROOT / "data" / "raw" / "dane"
CRS_GEO = "EPSG:4326"
CRS_COL = "EPSG:3116"

print(f"ROOT: {ROOT}")
print(f"IGAC raw dir: {DATA_RAW_IGAC}")
print(f"Existe directorio IGAC: {DATA_RAW_IGAC.exists()}")


In [ ]:
# ── Cargar predios IGAC ───────────────────────────────────────────────────
def cargar_predios(divipola: str, igac_dir: Path = DATA_RAW_IGAC) -> gpd.GeoDataFrame:
    """
    Carga el GeoPackage de predios del IGAC para el municipio indicado.
    Busca en data/raw/igac/predios/<DIVIPOLA>_predios.gpkg
    También acepta .shp y .geojson como formatos alternativos.
    """
    predios_dir = igac_dir / "predios"
    for ext in [".gpkg", ".shp", ".geojson", ".json"]:
        ruta = predios_dir / f"{divipola}_predios{ext}"
        if ruta.exists():
            print(f"[IGAC] Cargando predios desde: {ruta}")
            gdf = gpd.read_file(ruta).to_crs(CRS_GEO)
            print(f"[IGAC] {len(gdf):,} predios | columnas: {list(gdf.columns[:8])}")
            return gdf
    print(f"[IGAC] ADVERTENCIA: Archivo de predios NO encontrado para {divipola}")
    print(f"       Esperado en: {predios_dir / (divipola + '_predios.gpkg')}")
    print(f"       Descargue los datos desde https://geoportal.igac.gov.co/")
    return gpd.GeoDataFrame(
        columns=["CODIGO_PREDIO", "USO_SUELO", "AREA_TERRENO", "ESTRATO", "geometry"],
        geometry="geometry", crs=CRS_GEO
    )

predios_apartado = cargar_predios(DIVIPOLA_APARTADO)
print(f"\nPredios Apartadó: {len(predios_apartado)} registros")
if not predios_apartado.empty:
    display(predios_apartado.head())
else:
    print("GeoDataFrame vacío — cargue los datos del IGAC para continuar.")


In [ ]:
# ── Visualizar predios por USO_SUELO ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 10))

if predios_apartado.empty:
    ax.text(0.5, 0.5,
            "Sin datos IGAC\nDescargue predios desde:\nhttps://geoportal.igac.gov.co/",
            transform=ax.transAxes, ha="center", va="center", fontsize=14, color="gray",
            bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))
else:
    usos = predios_apartado["USO_SUELO"].fillna("Sin clasificar").unique()
    cmap_colors = plt.cm.get_cmap("tab20", len(usos))
    uso_color = {uso: cmap_colors(i) for i, uso in enumerate(usos)}
    predios_proj = predios_apartado.to_crs(CRS_COL)
    for uso, grupo in predios_proj.groupby(predios_proj["USO_SUELO"].fillna("Sin clasificar")):
        grupo.plot(ax=ax, color=uso_color[uso], alpha=0.7,
                   linewidth=0.3, edgecolor="white", label=uso)
    try:
        ctx.add_basemap(ax, crs=CRS_COL, source=ctx.providers.CartoDB.Positron, zoom=14)
    except Exception:
        pass
    ax.legend(loc="lower left", fontsize=7, title="Uso del Suelo", framealpha=0.9)

ax.set_title("Predios Apartadó — Clasificación por Uso del Suelo (IGAC)", fontsize=14, fontweight="bold")
ax.set_axis_off()
plt.tight_layout()
(ROOT / "data" / "outputs").mkdir(parents=True, exist_ok=True)
plt.savefig(ROOT / "data" / "outputs" / "igac_usosuelo_apartado.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Join predios con manzanas MGN → conteo por manzana ───────────────────
mgn_path = DATA_RAW_DANE / "MGN_MANZANA.shp"
if mgn_path.exists():
    from src.ingestion.dane_loader import cargar_mgn_manzanas
    manzanas_all = cargar_mgn_manzanas(mgn_path)
    manzanas_apartado = manzanas_all[
        manzanas_all["cod_manzana"].str.startswith(DIVIPOLA_APARTADO)
    ].copy()
    print(f"Manzanas Apartadó (MGN): {len(manzanas_apartado)}")
else:
    print("[DANE] MGN no encontrado — creando grid dummy para Apartadó")
    from shapely.geometry import box
    n = 20
    lons = np.linspace(-76.65, -76.61, n)
    lats = np.linspace(7.86, 7.90, n)
    geoms, codes = [], []
    for i, lon in enumerate(lons[:-1]):
        for j, lat in enumerate(lats[:-1]):
            geoms.append(box(lon, lat, lons[i+1], lats[j+1]))
            codes.append(f"05045{i:03d}{j:03d}")
    manzanas_apartado = gpd.GeoDataFrame(
        {"cod_manzana": codes, "poblacion": np.random.randint(50, 500, len(codes))},
        geometry=geoms, crs=CRS_GEO
    )
    print(f"Manzanas dummy: {len(manzanas_apartado)}")

# Join espacial
if not predios_apartado.empty:
    predios_proj = predios_apartado.to_crs(CRS_COL)
    manzanas_proj = manzanas_apartado.to_crs(CRS_COL)
    joined = gpd.sjoin(
        predios_proj[["geometry", "CODIGO_PREDIO"]],
        manzanas_proj[["cod_manzana", "geometry"]],
        how="left", predicate="within"
    )
    predios_por_manzana = joined.groupby("cod_manzana").size().rename("n_predios")
    manzanas_apartado = manzanas_apartado.merge(
        predios_por_manzana.reset_index(), on="cod_manzana", how="left"
    )
    manzanas_apartado["n_predios"] = manzanas_apartado["n_predios"].fillna(0).astype(int)
    print(f"Predios vinculados: {manzanas_apartado['n_predios'].sum():,}")
    display(manzanas_apartado[["cod_manzana", "n_predios"]].describe())
else:
    manzanas_apartado["n_predios"] = np.random.randint(5, 80, len(manzanas_apartado))
    print("Usando conteo dummy de predios.")


In [ ]:
# ── Visualizar densidad predial por manzana ───────────────────────────────
fig, ax = plt.subplots(figsize=(12, 10))
manzanas_proj = manzanas_apartado.to_crs(CRS_COL)
manzanas_proj.plot(
    column="n_predios",
    ax=ax,
    cmap="YlOrRd",
    legend=True,
    legend_kwds={"label": "Número de predios", "shrink": 0.7},
    edgecolor="white",
    linewidth=0.3,
    alpha=0.85
)
try:
    ctx.add_basemap(ax, crs=CRS_COL, source=ctx.providers.CartoDB.Positron)
except Exception:
    pass
ax.set_title("Densidad Predial por Manzana — Apartadó (IGAC + MGN)", fontsize=14, fontweight="bold")
ax.set_axis_off()
plt.tight_layout()
plt.savefig(ROOT / "data" / "outputs" / "igac_densidad_predial_apartado.png", dpi=150, bbox_inches="tight")
plt.show()
print("Mapa guardado en data/outputs/igac_densidad_predial_apartado.png")


In [ ]:
# ── Cargar edificaciones y probar dasymetría ──────────────────────────────
def cargar_edificaciones(divipola: str, igac_dir: Path = DATA_RAW_IGAC) -> gpd.GeoDataFrame:
    """
    Carga GeoPackage de edificaciones del IGAC para dasymetría.
    Busca en data/raw/igac/edificaciones/<DIVIPOLA>_edif.gpkg
    """
    edif_dir = igac_dir / "edificaciones"
    for ext in [".gpkg", ".shp", ".geojson"]:
        ruta = edif_dir / f"{divipola}_edif{ext}"
        if ruta.exists():
            print(f"[IGAC] Edificaciones desde: {ruta}")
            gdf = gpd.read_file(ruta).to_crs(CRS_GEO)
            print(f"[IGAC] {len(gdf):,} edificaciones")
            return gdf
    print(f"[IGAC] Edificaciones no encontradas para {divipola} — generando puntos dummy")
    from shapely.geometry import Point
    bbox = manzanas_apartado.total_bounds
    n_edif = 600
    xs = np.random.uniform(bbox[0], bbox[2], n_edif)
    ys = np.random.uniform(bbox[1], bbox[3], n_edif)
    return gpd.GeoDataFrame(
        {"PISOS": np.random.randint(1, 5, n_edif),
         "AREA_CONST": np.random.uniform(40, 300, n_edif)},
        geometry=[Point(x, y) for x, y in zip(xs, ys)],
        crs=CRS_GEO
    )

edificaciones = cargar_edificaciones(DIVIPOLA_APARTADO)
print(f"Edificaciones: {len(edificaciones):,} registros")

# Probar disaggregate_dasymetric con datos dummy
print("\nProbando desagregación dasimétrca...")
try:
    manzanas_proj = manzanas_apartado.to_crs(CRS_COL).copy()
    manzanas_proj["dummy_pop"] = np.random.randint(100, 1000, len(manzanas_proj))
    resultado = disaggregate_dasymetric(
        source=manzanas_proj,
        target=manzanas_proj,
        source_pop_col="dummy_pop",
        buildings=edificaciones.to_crs(CRS_COL)
    )
    print(f"Dasymetría OK: shape={resultado.shape}")
    display(resultado[["cod_manzana", "pop_disagg"]].head())
except Exception as e:
    print(f"disaggregate_dasymetric error: {e}")
    print("Verificar firma de función en src/processing/disaggregate.py")


## Conclusiones y próximos pasos

### Lo que logramos en este notebook
1. **Estructura de carga IGAC**: `cargar_predios()` busca en `data/raw/igac/predios/` y maneja
   la ausencia de archivos con mensajes informativos y GeoDataFrame vacío tipado.
2. **Join predios ↔ manzanas**: `sjoin` con predicado `within` cuenta predios por manzana del MGN,
   generando el indicador de densidad predial.
3. **Visualización por USO_SUELO**: mapa coroplético coloreado por categoría catastral IGAC.
4. **Densidad predial**: mapa de calor de predios por manzana para identificar zonas de alta densidad.
5. **Edificaciones para dasymetría**: `cargar_edificaciones()` alimenta `disaggregate_dasymetric`
   para refinar estimaciones de población sub-manzana.

### Próximos pasos
- [ ] Descargar datos reales IGAC desde https://geoportal.igac.gov.co/ para Apartadó y Turbo
- [ ] Calcular indicador de **mixtura de usos** (ratio residencial/comercial por manzana)
- [ ] Cruzar estrato socioeconómico catastral con variables CNPV 2018
- [ ] Integrar áreas de terreno como denominador para indicadores de aprovechamiento del suelo
- [ ] Continuar con notebook **04** para imágenes satelitales GEE (dimensión ambiental)
